# Intertidal mussel occurrence along Moroccan Mediterranean and Atlantic coasts Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In this notebook, we will:
- load the Croissant metadata,
- inspect dataset-level information,
- review available record sets using their `@id` values,
- safely attempt extraction of record data into pandas DataFrames,
- perform example exploratory data analysis when tabular record sets are available,
- and create basic visualizations.

Important: throughout the notebook, record sets, fields, and any other dataset entities are referenced by their `@id` values whenever available.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import json
import math
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.zbsf-jgwe/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))

In [ ]:
# Convert metadata to a JSON-like object only when needed for inspection.
# We still keep `dataset.metadata` itself as a single object.
metadata_json = dataset.metadata.to_json()

summary = {
    '@id': metadata_json.get('@id'),
    'name': metadata_json.get('name'),
    'identifier': metadata_json.get('identifier'),
    'version': metadata_json.get('version'),
    'license': metadata_json.get('license'),
    'datePublished': metadata_json.get('datePublished'),
    'temporalCoverage': metadata_json.get('temporalCoverage'),
    'spatialCoverage': metadata_json.get('spatialCoverage'),
    'recordSet_count': len(metadata_json.get('recordSet', [])),
    'distribution_count': len(metadata_json.get('distribution', [])),
    'visualization_count': len(metadata_json.get('visualization', [])),
}

pd.DataFrame([summary]).T.rename(columns={0: 'value'})

The dataset-level metadata already contains useful contextual information even before reading any records. This is particularly helpful for understanding sampling scope, time coverage, and documented limitations.

In [ ]:
# Inspect selected descriptive metadata fields.
descriptive_fields = [
    'keywords',
    'dataUseCases',
    'dataBiases',
    'dataLimitations',
    'dataCollectionType',
    'dataCollectionTimeframe'
]

for field_id in descriptive_fields:
    print(f'\n### {field_id}')
    value = metadata_json.get(field_id)
    pprint(value)


## 2. Data Overview
Review available record sets, fields, and their IDs.

This dataset package reports dataset-level metadata and distributions. We first inspect the available Croissant record sets using their `@id` values.

In [ ]:
record_sets_json = metadata_json.get('recordSet', [])
print(f'Number of record sets declared in metadata: {len(record_sets_json)}')

record_set_rows = []
for rs in record_sets_json:
    record_set_rows.append({
        'record_set_@id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description'),
        'field_count': len(rs.get('field', [])) if isinstance(rs.get('field', []), list) else 0,
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df if not record_sets_df.empty else pd.DataFrame({'message': ['No record sets were declared in the provided Croissant metadata.']})

If record sets are present, the next cell prints each record set and its fields using the corresponding `@id` values. If none are present, the notebook will continue with metadata-centered exploration and safe extraction patterns.

In [ ]:
if not record_sets_json:
    print('No record sets are available in this Croissant file.')
else:
    for rs in record_sets_json:
        record_set_id = rs.get('@id')
        print('\n' + '=' * 80)
        print('Record set @id:', record_set_id)
        print('Name:', rs.get('name'))
        print('Description:', rs.get('description'))
        print('Fields:')
        for field in rs.get('field', []):
            print({
                'field_@id': field.get('@id'),
                'name': field.get('name'),
                'description': field.get('description'),
                'dataType': field.get('dataType'),
                'source': field.get('source')
            })

The template below shows how records are accessed from a record set by `@id`. Since this specific metadata currently reports no record sets, the example is guarded to avoid runtime errors.

In [ ]:
if record_sets_json:
    example_record_set_id = record_sets_json[0].get('@id')
    print(f'Previewing records from record set: {example_record_set_id}')
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i >= 4:
            break
else:
    print('Example skipped because there is no record set @id to query in this Croissant file.')

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

This section is written dynamically so it works whether the metadata exposes zero, one, or multiple record sets.

In [ ]:
# Extract data from each record set using record set @id values.
record_set_ids = [rs.get('@id') for rs in record_sets_json if rs.get('@id')]
dataframes = {}
extraction_status = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        extraction_status.append({
            'record_set_@id': record_set_id,
            'loaded': True,
            'rows': len(df),
            'columns': len(df.columns)
        })
    except Exception as e:
        extraction_status.append({
            'record_set_@id': record_set_id,
            'loaded': False,
            'rows': None,
            'columns': None,
            'error': str(e)
        })

status_df = pd.DataFrame(extraction_status)
status_df if not status_df.empty else pd.DataFrame({'message': ['No record sets available for extraction.']})

In [ ]:
if dataframes:
    sample_record_set_id = next(iter(dataframes.keys()))
    print('Sample record set @id:', sample_record_set_id)
    print('Columns:')
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print('No tabular records were loaded. This is consistent with the current metadata, which declares no record sets.')

Even when no record sets are declared, we can still inspect distribution objects by `@id`, which often point to downloadable files or package components.

In [ ]:
distribution_rows = []
for dist in metadata_json.get('distribution', []):
    distribution_rows.append({
        'distribution_@id': dist.get('@id'),
        'name': dist.get('name'),
        'contentUrl': dist.get('contentUrl'),
        'encodingFormat': dist.get('encodingFormat'),
        'description': dist.get('description')
    })

pd.DataFrame(distribution_rows) if distribution_rows else pd.DataFrame({'message': ['No distribution entries found.']})

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Because this Croissant file currently exposes dataset-level metadata without record sets, this section provides two paths:
1. a general-purpose EDA workflow that runs if any tabular record set is available, and
2. a metadata-driven EDA summary that always works.

In [ ]:
# Helper: find a usable DataFrame and a numeric field automatically.
selected_record_set_id = None
selected_numeric_field_id = None
selected_group_field_id = None

for record_set_id, df in dataframes.items():
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    group_candidates = [c for c in df.columns if df[c].dtype == 'object' or str(df[c].dtype).startswith('string')]
    if numeric_candidates:
        selected_record_set_id = record_set_id
        selected_numeric_field_id = numeric_candidates[0]
        selected_group_field_id = group_candidates[0] if group_candidates else None
        break

print('selected_record_set_id =', selected_record_set_id)
print('selected_numeric_field_id =', selected_numeric_field_id)
print('selected_group_field_id =', selected_group_field_id)

In [ ]:
if selected_record_set_id is not None and selected_numeric_field_id is not None:
    working_df = dataframes[selected_record_set_id].copy()
    numeric_field = selected_numeric_field_id
    group_field = selected_group_field_id

    threshold = working_df[numeric_field].median()
    filtered_df = working_df[working_df[numeric_field] > threshold].copy()

    print(f'Filtered records with {numeric_field} > {threshold}:')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
    else:
        filtered_df[f'{numeric_field}_normalized'] = 0.0

    print(f'Normalized {numeric_field} for filtered records:')
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].mean().reset_index()
        print(f'Grouped data by {group_field}:')
        display(grouped_df.head())
else:
    print('Tabular EDA skipped because no record set with numeric fields was available.')

In [ ]:
# Metadata-driven EDA that always works for this package.
metadata_eda = pd.DataFrame([
    {'metric': 'keyword_count', 'value': len(metadata_json.get('keywords', []))},
    {'metric': 'dataUseCases_count', 'value': len(metadata_json.get('dataUseCases', []))},
    {'metric': 'dataBiases_count', 'value': len(metadata_json.get('dataBiases', []))},
    {'metric': 'dataLimitations_count', 'value': len(metadata_json.get('dataLimitations', []))},
    {'metric': 'distribution_count', 'value': len(metadata_json.get('distribution', []))},
    {'metric': 'recordSet_count', 'value': len(metadata_json.get('recordSet', []))},
    {'metric': 'visualization_count', 'value': len(metadata_json.get('visualization', []))},
    {'metric': 'author_count', 'value': len(metadata_json.get('author', []))},
    {'metric': 'citation_count', 'value': len(metadata_json.get('citation', []))},
    {'metric': 'funding_count', 'value': len(metadata_json.get('funding', []))},
])

metadata_eda

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will generate:
- a metadata-level bar chart summarizing package contents, and
- optional tabular plots if any record sets are successfully loaded.

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = metadata_eda.copy()
sns.barplot(data=plot_df, x='metric', y='value', palette='viridis')
plt.title('Metadata-level summary of the FAIR2 Croissant package')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
if selected_record_set_id is not None and selected_numeric_field_id is not None:
    working_df = dataframes[selected_record_set_id].copy()
    numeric_field = selected_numeric_field_id

    plt.figure(figsize=(8, 5))
    sns.histplot(working_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field} in record set {selected_record_set_id}')
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if selected_group_field_id is not None and selected_group_field_id in working_df.columns:
        top_groups = working_df[selected_group_field_id].astype(str).value_counts().head(10).index.tolist()
        grouped_plot_df = working_df[working_df[selected_group_field_id].astype(str).isin(top_groups)].copy()

        plt.figure(figsize=(10, 5))
        sns.boxplot(data=grouped_plot_df, x=selected_group_field_id, y=numeric_field)
        plt.title(f'{numeric_field} by {selected_group_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('Tabular visualizations skipped because no record-set table was available.')

## 6. Conclusion
This exploration showed how to work with a Croissant dataset using `mlcroissant` while consistently referencing dataset entities by `@id`.

Key observations:
- The dataset metadata loaded successfully from the Croissant URL.
- The package provides rich dataset-level FAIR and documentation metadata.
- In the provided Croissant file, the `recordSet` collection is currently empty, so no tabular records were directly extracted through `dataset.records(record_set=<@id>)`.
- The package still exposes useful information through high-level metadata and distribution objects identified by `@id`.

If a future version of the Croissant file includes populated record sets, the same notebook structure will automatically support:
- listing record set `@id` values,
- loading rows into pandas DataFrames,
- selecting fields by `@id`,
- applying filtering and normalization,
- and creating record-level visualizations.

This makes the notebook a reusable starting point for Croissant-based dataset exploration.